# Write Tags to Database

Write predicted tags from inference results to the RekordBox database.

**IMPORTANT:** Make sure to create a backup before editing the DB. The backup dialog can be found under "File" > "Library" > "Backup Library".

This notebook:
1. Loads tag predictions from the parquet file
2. Writes tags to the database in batches, grouped by song
3. Commits to the database after each batch (finishing all tags for the current song)

In [1]:
# automatically reload imported modules before executing code
%load_ext autoreload
%autoreload 2

In [2]:
from nbutils import setup_path, display_polars
import polars as pl
from pathlib import Path

setup_path()

In [3]:
from pyrekordbox import Rekordbox6Database

db = Rekordbox6Database()

## Configuration

In [4]:
# Path to the inference results parquet file
INFERENCE_RESULTS_PATH = "../data/full_inference_results_20251018_1440.parquet"

# Batch size: commit to database after this many tags
# Important: we finish all tags for the current song before committing,
# even if it exceeds this number
TAG_COMMIT_BATCH_SIZE = 100

print(f"Configuration:")
print(f"  Input file: {INFERENCE_RESULTS_PATH}")
print(f"  Batch size: {TAG_COMMIT_BATCH_SIZE} tags (per batch, completing current song)")

Configuration:
  Input file: ../data/full_inference_results_20251018_1440.parquet
  Batch size: 100 tags (per batch, completing current song)


## Load Predictions

In [5]:
# Load the predictions
predictions_df = pl.read_parquet(INFERENCE_RESULTS_PATH)

print(f"Loaded predictions:")
print(f"  Total rows: {len(predictions_df)}")
print(f"  Unique songs: {predictions_df.select('song_id').n_unique()}")
print(f"\nColumns: {predictions_df.columns}")
print(f"\nFirst 10 rows:")
display_polars(predictions_df.head(10))

Loaded predictions:
  Total rows: 3064
  Unique songs: 724

Columns: ['song_id', 'song_title', 'artist_name', 'tag', 'tag_id', 'tag_uuid', 'tag_group']

First 10 rows:
shape: (10, 7)
┌───────────┬───────────────┬───────────────┬──────────────┬────────────┬──────────────┬───────────┐
│ song_id   ┆ song_title    ┆ artist_name   ┆ tag          ┆ tag_id     ┆ tag_uuid     ┆ tag_group │
│ ---       ┆ ---           ┆ ---           ┆ ---          ┆ ---        ┆ ---          ┆ ---       │
│ str       ┆ str           ┆ str           ┆ str          ┆ i64        ┆ str          ┆ str       │
╞═══════════╪═══════════════╪═══════════════╪══════════════╪════════════╪══════════════╪═══════════╡
│ 161119069 ┆ Good          ┆ Session       ┆ Ambient      ┆ 2484825285 ┆ 22a59e2a-b3a ┆ Genre     │
│           ┆ Intentions    ┆ Victim        ┆              ┆            ┆ 6-470e-ad4b- ┆           │
│           ┆               ┆               ┆              ┆            ┆ 92eddadcea7b ┆           │
│ 1611190

## Verify Tags Exist in Database

Check that all tag IDs in the predictions exist in the database.

In [6]:
# Get unique tag IDs from predictions
unique_tag_ids = predictions_df.select('tag_id').unique().to_series().to_list()

print(f"Checking {len(unique_tag_ids)} unique tags in database...")

# Check each tag exists
missing_tags = []
for tag_id in unique_tag_ids:
    try:
        tag = db.get_my_tag(ID=str(tag_id))
    except Exception as e:
        missing_tags.append(tag_id)

if missing_tags:
    print(f"\n⚠️  WARNING: {len(missing_tags)} tags not found in database:")
    for tag_id in missing_tags:
        matching_rows = predictions_df.filter(pl.col('tag_id') == tag_id)
        if len(matching_rows) > 0:
            tag_name = matching_rows.select('tag').item(0)
            print(f"  - Tag ID {tag_id}: {tag_name}")
    print("\nPlease create these tags in RekordBox before proceeding.")
else:
    print("✓ All tags exist in database")

Checking 56 unique tags in database...
✓ All tags exist in database


## Verify Songs Exist in Database

Check that all song IDs in the predictions exist in the database.

In [7]:
# Get unique song IDs from predictions
unique_song_ids = predictions_df.select('song_id').unique().to_series().to_list()

print(f"Checking {len(unique_song_ids)} unique songs in database...")

# Check each song exists
missing_songs = []
for song_id in unique_song_ids:
    try:
        content = db.get_content(ID=str(song_id))
    except Exception as e:
        missing_songs.append(song_id)

if missing_songs:
    print(f"\n⚠️  WARNING: {len(missing_songs)} songs not found in database:")
    for song_id in missing_songs[:10]:  # Show first 10
        matching_rows = predictions_df.filter(pl.col('song_id') == song_id)
        if len(matching_rows) > 0:
            song_title = matching_rows.select('song_title').item(0)
            print(f"  - Song ID {song_id}: {song_title}")
    if len(missing_songs) > 10:
        print(f"  ... and {len(missing_songs) - 10} more")
    print("\nThese songs may have been deleted from your RekordBox library.")
else:
    print("✓ All songs exist in database")

Checking 724 unique songs in database...
✓ All songs exist in database


## Write Tags to Database

Write tags in batches, ensuring we complete all tags for a song before committing.

In [9]:
from utils import add_tag
from tqdm.auto import tqdm

# Group predictions by song_id to ensure we process all tags for a song together
song_groups = predictions_df.partition_by('song_id', as_dict=True)

print(f"Writing tags to database...")
print(f"Total songs: {len(song_groups)}")
print(f"Total tags to write: {len(predictions_df)}")
print(f"Batch size: {TAG_COMMIT_BATCH_SIZE} (will finish current song before committing)")
print("\n" + "="*80)

tags_written = 0
songs_processed = 0
batches_committed = 0
errors = []

# Create progress bar
pbar = tqdm(total=len(predictions_df), desc="Writing tags", unit="tags")

for song_id, song_df in song_groups.items():
    song_title = song_df.select('song_title').item(row=0, column=0)
    artist_name = song_df.select('artist_name').item(row=0, column=0)
    
    # Process all tags for this song
    song_tags_written = 0
    
    for row in song_df.iter_rows(named=True):
        tag_id = str(row['tag_id'])
        content_id = str(row['song_id'])
        tag_name = row['tag']
        
        try:
            # Add tag to database (doesn't commit yet)
            add_tag(db, tag_id=tag_id, content_id=content_id)
            tags_written += 1
            song_tags_written += 1
            pbar.update(1)
            
        except ValueError as e:
            # Tag already exists, skip
            if "already associated" in str(e):
                pbar.update(1)
                continue
            else:
                error_msg = f"Error adding tag '{tag_name}' to '{song_title}': {e}"
                errors.append(error_msg)
                pbar.update(1)
                
        except Exception as e:
            error_msg = f"Error adding tag '{tag_name}' to '{song_title}': {e}"
            errors.append(error_msg)
            pbar.update(1)
    
    songs_processed += 1
    
    # Check if we should commit (either reached batch size or this is the last song)
    # We only commit after finishing all tags for the current song
    is_last_song = songs_processed == len(song_groups)
    exceeded_batch = tags_written >= TAG_COMMIT_BATCH_SIZE
    
    if exceeded_batch or is_last_song:
        db.commit()
        batches_committed += 1
        pbar.set_description(f"Writing tags (committed {batches_committed} batches)")
        tags_written = 0  # Reset counter for next batch

pbar.close()

print("\n" + "="*80)
print(f"\n✓ Tag writing complete!")
print(f"  Songs processed: {songs_processed}")
print(f"  Batches committed: {batches_committed}")

if errors:
    print(f"\n⚠️  Encountered {len(errors)} errors:")
    for error in errors[:10]:  # Show first 10 errors
        print(f"  - {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more errors")

Writing tags to database...
Total songs: 724
Total tags to write: 3064
Batch size: 100 (will finish current song before committing)



Writing tags:   0%|          | 0/3064 [02:16<?, ?tags/s]


Adding tag with ID=2484825285 to content with ID=161119069
Content ID:  161119069
Tag ID:      2484825285
ID:          c6d4b3e7-9c42-4140-a613-cb1d801d43e6
UUID:        7b217247-d86e-4fb3-aa64-ee41e2a126e8
TrackNo:     2
Adding tag with ID=2333129259 to content with ID=161119069
Content ID:  161119069
Tag ID:      2333129259
ID:          e6e4e4e5-637c-4fe4-866b-bce46e380529
UUID:        2f445cf9-b290-49f1-b26a-4efab38822f9
TrackNo:     3
Adding tag with ID=921280454 to content with ID=161119069
Content ID:  161119069
Tag ID:      921280454
ID:          b9c5a0f6-fe54-4fbc-bf10-ba15082e3ad8
UUID:        7f00b2b0-ac91-459e-a970-24c02c7f6be7
TrackNo:     4
Adding tag with ID=1440793859 to content with ID=161119069
Content ID:  161119069
Tag ID:      1440793859
ID:          1100db11-0376-4d26-831b-65d93dce3e29
UUID:        93cc287a-fb38-443a-b858-a0eb866de138
TrackNo:     5
Adding tag with ID=2813249465 to content with ID=260809205
Content ID:  260809205
Tag ID:      2813249465
ID:         

Adding tag with ID=2688783057 to content with ID=51117658
Content ID:  51117658
Tag ID:      2688783057
ID:          f6c6e0ea-6a29-4659-999d-e9e5ca84fc95
UUID:        872d2f3f-148b-4d56-9bdf-2890a38c7d1e
TrackNo:     2
Adding tag with ID=1773027948 to content with ID=51117658
Content ID:  51117658
Tag ID:      1773027948
ID:          6397aee7-5d02-43d1-a026-087a9ce7203c
UUID:        7f894ae3-9ed5-4816-9bbd-5cf2eb96d426
TrackNo:     3
Adding tag with ID=378434426 to content with ID=51117658
Content ID:  51117658
Tag ID:      378434426
ID:          9ba1dec0-bc71-4226-a920-b19c137d1ef7
UUID:        6dae800e-c4de-4f45-83db-68830dfdd0d7
TrackNo:     4
Adding tag with ID=921280454 to content with ID=51117658
Content ID:  51117658
Tag ID:      921280454
ID:          51995a27-ce8b-406a-aff8-5c13c276cbbb
UUID:        d640121d-e45e-4de5-935e-edc203288ab1
TrackNo:     5
Adding tag with ID=165295017 to content with ID=51117658
Content ID:  51117658
Tag ID:      165295017
ID:          e584ac5f-73d5

Adding tag with ID=165295017 to content with ID=103635158
Content ID:  103635158
Tag ID:      165295017
ID:          50279b7e-aeb5-4656-8c1b-3d4b1e9b5ad2
UUID:        839ac3ca-4825-42bb-9d8e-6de4f358e88d
TrackNo:     3
Adding tag with ID=2445755319 to content with ID=103635158
Content ID:  103635158
Tag ID:      2445755319
ID:          448fa106-ff78-4876-ba78-9e0826846114
UUID:        1d507c6d-c946-4f6a-b882-0e682ce55eac
TrackNo:     4
Adding tag with ID=1440793859 to content with ID=103635158
Content ID:  103635158
Tag ID:      1440793859
ID:          247333d2-616b-4c65-8846-c7803ec42623
UUID:        8a91e411-2f95-49b2-8328-58fd5e652af9
TrackNo:     5
Adding tag with ID=2813249465 to content with ID=241730285
Content ID:  241730285
Tag ID:      2813249465
ID:          a80bd24d-38ec-453b-a69a-4f489817b674
UUID:        34803ae6-034a-476f-875c-8b91f1f692a3
TrackNo:     2
Adding tag with ID=378434426 to content with ID=241730285
Content ID:  241730285
Tag ID:      378434426
ID:          c

...

Adding tag with ID=2829126053 to content with ID=29110294
Content ID:  29110294
Tag ID:      2829126053
ID:          bc2742b0-4d17-4868-b5a1-cc0cc2976ebd
UUID:        34a3279f-5eab-40ac-9841-52fe715b4e12
TrackNo:     6
Adding tag with ID=62609778 to content with ID=191218482
Content ID:  191218482
Tag ID:      62609778
ID:          65632aa1-6e48-4a46-ba00-6de9bba21c71
UUID:        2031857a-2812-4a98-a723-6d4289d6c5ea
TrackNo:     2
Adding tag with ID=385085509 to content with ID=191218482
Content ID:  191218482
Tag ID:      385085509
ID:          f21fe725-e6c6-4746-900a-83c928a2d9f3
UUID:        13daa029-71a7-425b-9ac5-71a1565f96b0
TrackNo:     3
Adding tag with ID=2813249465 to content with ID=191218482
Content ID:  191218482
Tag ID:      2813249465
ID:          b8d4525c-10f3-4da4-b7f8-a1dee479fb4d
UUID:        c68da3c9-6651-403d-935a-7c8a34a0f403
TrackNo:     4
Adding tag with ID=378434426 to content with ID=191218482
Content ID:  191218482
Tag ID:      378434426
ID:          79abbd4

Adding tag with ID=1488675436 to content with ID=138029452
Content ID:  138029452
Tag ID:      1488675436
ID:          a999f680-ec7c-4d55-9ba7-bbc1a3895045
UUID:        9afd0962-5072-413c-9717-f09984da05de
TrackNo:     2
Adding tag with ID=378434426 to content with ID=138029452
Content ID:  138029452
Tag ID:      378434426
ID:          ed9cd018-2d5d-408a-8951-dc76d2208835
UUID:        afada12f-5646-47e1-bdf7-308b7d333dc6
TrackNo:     3
Adding tag with ID=921280454 to content with ID=138029452
Content ID:  138029452
Tag ID:      921280454
ID:          4350c4cb-3355-4e53-989d-cba8ba6f280b
UUID:        f914b565-a23a-4f8a-af2b-36343f1c4c10
TrackNo:     4
Adding tag with ID=2445755319 to content with ID=138029452
Content ID:  138029452
Tag ID:      2445755319
ID:          c799c90c-6534-4e19-9cd4-898c0b04d3bd
UUID:        d27e36e3-e5b0-4610-817a-8757373f1429
TrackNo:     5
Adding tag with ID=681822852 to content with ID=142687956
Content ID:  142687956
Tag ID:      681822852
ID:          e7b

Adding tag with ID=2355752193 to content with ID=2647934
Content ID:  2647934
Tag ID:      2355752193
ID:          c421aec3-0bbd-4f31-b0af-27635155a0d6
UUID:        70d31d70-26f5-46bc-b0da-7fd71898f726
TrackNo:     6
Adding tag with ID=2688783057 to content with ID=79923981
Content ID:  79923981
Tag ID:      2688783057
ID:          34034d4d-b4f7-4e06-89e2-f4365d67d583
UUID:        6fef5289-9faf-4f81-b22a-53745f6335fb
TrackNo:     2
Adding tag with ID=3531125754 to content with ID=79923981
Content ID:  79923981
Tag ID:      3531125754
ID:          0467cc90-fa49-4199-9c67-e438970558a6
UUID:        4c011751-c487-4ead-b9b6-972a5b990af5
TrackNo:     3
Adding tag with ID=165295017 to content with ID=79923981
Content ID:  79923981
Tag ID:      165295017
ID:          67939c18-1bca-4c56-bb15-a25c267a354e
UUID:        e5a698e7-9bf7-44e7-8898-930dc136a8cc
TrackNo:     4
Adding tag with ID=681822852 to content with ID=124360475
Content ID:  124360475
Tag ID:      681822852
ID:          cb19b811-dc

Writing tags (committed 30 batches): 100%|██████████| 3064/3064 [00:04<00:00, 724.02tags/s]

Adding tag with ID=921280454 to content with ID=135125269
Content ID:  135125269
Tag ID:      921280454
ID:          69787547-2204-4638-a236-487cc62909ce
UUID:        a9c2ff14-3b36-4b82-86e5-94964469e658
TrackNo:     2
Adding tag with ID=378434426 to content with ID=260225544
Content ID:  260225544
Tag ID:      378434426
ID:          f4301544-c662-4681-8f4f-8ec50c2660d1
UUID:        f4c45a1c-5907-4f3e-b4f0-11487a13033f
TrackNo:     2
Adding tag with ID=1440793859 to content with ID=260225544
Content ID:  260225544
Tag ID:      1440793859
ID:          15ca26c4-5f3b-4b93-b38b-9ecd2ea339fa
UUID:        c2ff58c7-3cd2-48dd-8f33-2c249d6799c3
TrackNo:     3
Adding tag with ID=1331247722 to content with ID=266374668
Content ID:  266374668
Tag ID:      1331247722
ID:          7a983a41-9072-4960-a62b-5f41bbf2ebb0
UUID:        679a260d-f776-4b23-95f9-4f7df3b92cb5
TrackNo:     2
Adding tag with ID=921280454 to content with ID=266374668
Content ID:  266374668
Tag ID:      921280454
ID:          34c

## Cleanup

Close the database session.

In [10]:
# Remove any session if it exists
if db.session:
    db.session.close()
    print("Database session closed")

Database session closed


## Summary

Tags have been written to the RekordBox database!

The process:
1. ✓ Loaded predictions from parquet file
2. ✓ Verified all tags and songs exist in database
3. ✓ Wrote tags in batches (grouped by song)
4. ✓ Committed to database after each batch

The tags should now be visible in RekordBox.